## Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text preprocessing
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer

# Model
from sklearn.linear_model import Perceptron

# Model evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

## Load the Dataset

In [ ]:
df = pd.read_csv("spam_Emails_data.csv")

In [ ]:
df.head()

## Understand the Dataset

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df.sample(5)

## Data Quality Check

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

## Visualization

In [ ]:
sns.countplot(x="label", data=df)
plt.title("Distribution of Ham and Spam Emails")
plt.show()

## Exploratory Data Analysis (EDA)

## Dataset Shape

In [ ]:
print("Rows and Columns:", df.shape)

## Dataset Information

In [ ]:
df.info()

## Missing Values

In [ ]:
df.isnull().sum()

## Duplicate Emails

In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

## Target Variable Distribution

In [ ]:
class_counts = df["label"].value_counts()
print(class_counts)

In [ ]:
class_percent = df["label"].value_counts(normalize=True) * 100
print(class_percent)

Visualization

In [ ]:
plt.figure(figsize=(6,5))

sns.countplot(
    data=df,
    x="label",
    palette="Set2"
)

plt.title("Distribution of Ham and Spam Emails")
plt.xlabel("Email Type")
plt.ylabel("Number of Emails")
plt.xticks([0,1],["Ham","Spam"])

plt.show()

## Email Length Analysis

In [ ]:
df = df.dropna(subset=["text"])

In [ ]:
df["email_length"] = df["text"].apply(len)

In [ ]:
df["text"].isnull().sum()

In [ ]:
df["text"].dtype

In [ ]:
df["email_length"].describe()

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    data=df,
    x="email_length",
    bins=50,
    kde=True
)

plt.title("Distribution of Email Length")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(
    data=df,
    x="label",
    y="email_length",
    palette="Set3"
)

plt.xticks([0,1],["Ham","Spam"])

plt.title("Email Length by Class")

plt.show()

In [ ]:
df["word_count"] = df["text"].apply(lambda x: len(str(x).split()))

In [ ]:
df.head()

Summary

In [ ]:
df["word_count"].describe()

Histogram

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    df["word_count"],
    bins=50,
    color="green",
    kde=True
)

plt.title("Word Count Distribution")

plt.show()

## Compare Word Counts

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(
    data=df,
    x="label",
    y="word_count",
    palette="Pastel1"
)

plt.xticks([0,1],["Ham","Spam"])

plt.title("Word Count by Email Type")

plt.show()

## Character Count

In [ ]:
df["character_count"] = df["text"].apply(len)

In [ ]:
df.groupby("label")["character_count"].mean()

## Average Statistics

In [ ]:
df.groupby("label")[["email_length","word_count"]].mean()

## Correlation Heatmap

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    df[["email_length","word_count","character_count"]].corr(),
    annot=True,
    cmap="Blues"
)

plt.title("Correlation Between Numerical Features")

plt.show()

## Text Preprocessing

## Required Libraries

In [ ]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

## Download NLTK Resources

In [ ]:
nltk.download("stopwords")

## Create a Stemmer

In [ ]:
stemmer = PorterStemmer()

## Stop Words

In [ ]:
stop_words = set(stopwords.words("english"))

## Create the Cleaning Function

In [ ]:
import re
import string

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()


def clean_text(text):

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove standalone http, https, www
    text = re.sub(r'\b(http|https|www)\b', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove dataset placeholders
    text = re.sub(r'escapenumber\w*', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove stopwords and apply stemming
    words = [
        stemmer.stem(word)
        for word in text.split()
        if word not in stop_words
    ]

    return " ".join(words)

## Apply Cleaning

In [ ]:
df["clean_text"] = df["text"].apply(clean_text)

## Compare Before and After

In [ ]:
df[["text","clean_text"]].head(10)

## Check Empty Emails

In [ ]:
(df["clean_text"] == "").sum()

## Compare Word Counts Before and After

In [ ]:
df["clean_word_count"] = df["clean_text"].str.split().str.len()

df[["word_count","clean_word_count"]].describe()

In [ ]:
df[["text", "clean_text"]].head(10)

In [ ]:
df["clean_text"].sample(5)

## Compare Word Counts Before vs After Cleaning

In [ ]:
df["clean_word_count"] = df["clean_text"].str.split().str.len()

df[["word_count", "clean_word_count"]].describe()

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    df["clean_word_count"],
    bins=50,
    kde=True,
    color="green"
)

plt.title("Distribution of Cleaned Word Count")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.show()

## TF-IDF Vectorization

## Import TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

## Create the Vectorizer

In [ ]:
X = tfidf.fit_transform(df["clean_text"])

## Target Variable

In [ ]:
y = df["label"]

## Check the Shapes

In [ ]:
print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

## View the Vocabulary

In [ ]:
print(tfidf.get_feature_names_out()[:20])

## Convert to a DataFrame

In [ ]:
tfidf_df = pd.DataFrame(
    X[:5].toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df.head()

## Storytelling

## Check for Sparsity

In [ ]:
print(type(X))

## Split the Dataset

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Verify the Split

In [ ]:
print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("Training Labels:", y_train.shape)
print("Testing Labels:", y_test.shape)

## Build the Perceptron Model

## Import the Perceptron Classifier

In [ ]:
from sklearn.linear_model import Perceptron

## Create the Model

In [ ]:
model = Perceptron(
    max_iter=1000,
    random_state=42,
    eta0=1.0
)

## Train the Model

In [ ]:
model.fit(X_train, y_train)

## Make Predictions

In [ ]:
y_pred = model.predict(X_test)

## Evaluate Accuracy

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

## Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

## Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

## Visualize the Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

plt.figure(figsize=(6,5))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Ham", "Spam"]
).plot(cmap="Blues")

plt.title("Confusion Matrix")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

metrics = {
    "Accuracy": 0.9672,
    "Precision": 0.97,
    "Recall": 0.97,
    "F1-score": 0.97
}

plt.figure(figsize=(8, 5))

plt.bar(metrics.keys(), metrics.values())

plt.ylim(0.9, 1.0)

plt.title("Perceptron Model Performance")
plt.ylabel("Score")

for i, value in enumerate(metrics.values()):
    plt.text(i, value + 0.002, f"{value:.3f}", ha="center")

plt.show()

## Compare Actual vs Predicted

In [ ]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

results.head(10)

## Predict a New Email

In [ ]:
sample_email = [
    "Congratulations! You have won a free iPhone. Click here to claim your prize."
]

In [ ]:
clean_sample = [clean_text(sample_email[0])]

In [ ]:
sample_vector = tfidf.transform(clean_sample)

Predict:

In [ ]:
def predict_email(email):

    # Step 1: Clean the email
    cleaned = clean_text(email)

    # Step 2: Convert to TF-IDF
    vector = tfidf.transform([cleaned])

    # Step 3: Predict
    prediction = model.predict(vector)[0]

    print("Original Email:")
    print(email)
    print()

    print("Prediction:", prediction)

    if prediction == "Spam":
        print("🚨 This email is classified as SPAM.")
    else:
        print("✅ This email is classified as HAM.")

In [ ]:
predict_email(
    "Hi John, can we meet tomorrow at 2 PM to discuss the project?"
)

In [ ]:
predict_email(
    "Exclusive offer! Buy one get one free. Limited time only."
)

In [ ]:
predict_email(
    "Please find attached the meeting agenda for next week's discussion."
)

## Feature Importance

## Extract Feature Weights

In [ ]:
import pandas as pd

feature_names = tfidf.get_feature_names_out()

weights = model.coef_[0]

feature_importance = pd.DataFrame({
    "Word": feature_names,
    "Weight": weights
})

feature_importance.head()

## Top Spam Words

In [ ]:
top_spam = feature_importance.sort_values(
    by="Weight",
    ascending=False
).head(20)

top_spam

In [ ]:
plt.figure(figsize=(10,6))

plt.barh(
    top_spam["Word"],
    top_spam["Weight"]
)

plt.title("Top 20 Words Associated with Spam")
plt.xlabel("Perceptron Weight")
plt.gca().invert_yaxis()

plt.show()

## Top Ham Words

In [ ]:
top_ham = feature_importance.sort_values(
    by="Weight",
    ascending=True
).head(20)

top_ham

In [ ]:
plt.figure(figsize=(10,6))

plt.barh(
    top_ham["Word"],
    top_ham["Weight"]
)

plt.title("Top 20 Words Associated with Ham")
plt.xlabel("Perceptron Weight")
plt.gca().invert_yaxis()

plt.show()

## Word Clouds

In [ ]:
from wordcloud import WordCloud

## Ham Word Cloud

In [ ]:
ham_text = " ".join(
    df[df["label"]=="Ham"]["clean_text"]
)

wordcloud = WordCloud(
    width=900,
    height=500,
    background_color="white"
).generate(ham_text)

plt.figure(figsize=(12,6))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("Most Common Words in Ham Emails")

plt.show()

## Spam Word Cloud

In [ ]:
spam_text = " ".join(
    df[df["label"]=="Spam"]["clean_text"]
)

wordcloud = WordCloud(
    width=900,
    height=500,
    background_color="white"
).generate(spam_text)

plt.figure(figsize=(12,6))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("Most Common Words in Spam Emails")

plt.show()

## Test Multiple Emails

In [ ]:
emails = [
    "Meeting has been moved to 2 PM tomorrow.",
    "Earn money fast with this amazing opportunity.",
    "Please review the attached report before today's meeting."
]

for email in emails:

    cleaned = clean_text(email)

    vector = tfidf.transform([cleaned])

    prediction = model.predict(vector)[0]

    print("-"*60)
    print(email)
    print("Prediction:", prediction)